In [1]:
!pip install PyMuPDF
!pip install nltk pymupdf
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 33.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu1

In [2]:
# Importing liabraries
import fitz  # PyMuPDF for PDF text extraction
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [6]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# Download necessary resources (run once)
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download('punkt_tab')
# nltk.download('punkt')
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


**Preprocessing the PDF Text into form understand by Machines**

In [8]:
# Function to clean the text extracted from PDF
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation & special characters
    tokens = word_tokenize(text)  # Tokenize text into words
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)  # Convert tokens back to string

# Function to extract the text from PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
    return clean_text(text)

In [9]:
# Uploading two documents which are supposed to be compared
#If you don't want to upload your own document i have attach a sample doc content below
path1 = "/content/pdf1.pdf"  # Replace with your file path
path2 = "/content/pdf2.pdf"  # Replace with your file path
file1 = extract_text_from_pdf(path1)
file2 = extract_text_from_pdf(path2)

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


**Defining Functions which are needed to compare two documents either similar or not**

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load Sentence Transformer model for similarity detection
model = SentenceTransformer('all-mpnet-base-v2')

# Load T5 model for summarization
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
tokenizer = AutoTokenizer.from_pretrained("t5-small")

# Function to get weighted document embedding
def get_weighted_document_embedding(doc, model, chunk_size=100):
    words = doc.split()  # Split by whitespace
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

    embeddings = model.encode(chunks)  # Get chunk embeddings
    weights = np.linspace(1, 2, len(embeddings))  # Assign increasing weights
    weighted_embedding = np.average(embeddings, axis=0, weights=weights)

    return weighted_embedding, chunks, embeddings  # Return embeddings & original chunks

# Function to extract key overlapping content
def extract_overlapping_content(doc1_chunks, doc1_embeddings, doc2_chunks, doc2_embeddings, top_n=5):
    similarities = cosine_similarity(doc1_embeddings, doc2_embeddings)  # Compare all chunks
    top_indices = np.unravel_index(np.argsort(similarities.ravel())[-top_n:], similarities.shape)  # Get top matches

    overlapping_text = []
    for i, j in zip(top_indices[0], top_indices[1]):
        overlapping_text.append(doc1_chunks[i])

    return "\n".join(overlapping_text)  # Return joined similar sections

# Function to summarize overlapping content
def summarize_similarity(overlapping_content):
    if not overlapping_content.strip():
        return "There is no significant overlap between the two documents."

    prompt = "Summarize the following overlapping research content:\n" + overlapping_content
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    summary_ids = summarizer_model.generate(inputs.input_ids, max_length=100, num_beams=5, early_stopping=True)

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Function to compare two documents and generate a meaningful summary
def compare_and_summarize(doc1, doc2):
    emb1, doc1_chunks, doc1_embeddings = get_weighted_document_embedding(doc1, model)
    emb2, doc2_chunks, doc2_embeddings = get_weighted_document_embedding(doc2, model)

    # Compute document-level similarity
    similarity_score = cosine_similarity([emb1], [emb2])[0][0]

    # Extract overlapping content
    mutual_content = extract_overlapping_content(doc1_chunks, doc1_embeddings, doc2_chunks, doc2_embeddings)

    # Summarize the mutual content
    summarized_overlap = summarize_similarity(mutual_content)

    # Generate final report
    summary = f"🔍 **Similarity Score:** {similarity_score:.4f}\n\n"
    if similarity_score > 0.75:
        summary += "✅ **Your submitted research is highly similar to an existing paper.**\n"
        summary += "📄 **Summary of the overlap:**\n" + summarized_overlap
        summary += "\n⚠️ **Consider revising your research to provide new insights.**"
    elif similarity_score > 0.50:
        summary += "⚠️ **Your submission has some overlapping content with prior research.**\n"
        summary += "📄 **Summary of the overlap:**\n" + summarized_overlap
        summary += "\n✔️ **You might want to refine your approach to introduce more unique aspects.**"
    else:
        summary += "✔️ **Your research appears unique with minimal overlap.**\n"
        summary += "🎉 **Proceed with confidence!**"

    return summary

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [11]:
# Sample Documents to check the functionality
doc1 = """Artificial intelligence, particularly deep learning, has transformed various
industries by enabling machines to learn from vast amounts of data. Using artificial neural
networks, deep learning models can recognize patterns, make predictions, and automate complex
tasks in fields such as healthcare, finance, and autonomous systems. The rapid growth in
computational power and availability of large datasets have significantly advanced AI capabilities,
making it a crucial part of modern technology."""

doc2 = """The rise of deep learning has revolutionized artificial intelligence by allowing
machines to process and analyze large datasets more efficiently. Leveraging artificial neural
networks, these models can detect patterns, generate predictions, and streamline automation
across sectors like healthcare, finance, and self-driving technology. With increased computational
resources and extensive data, AI has evolved into a key driver of technological progress."""

In [12]:
# Run similarity check
result_summary = compare_and_summarize(doc1, doc2) # result_summary = compare_and_summarize(file1, file2)

# Print the summary
print(result_summary)

🔍 **Similarity Score:** 0.9467

✅ **Your submitted research is highly similar to an existing paper.**
📄 **Summary of the overlap:**
the following research content:: Summarize the following research content: Artificial intelligence has transformed various industries by enabling machines to learn from vast amounts of data. Using artificial neural networks, deep learning models can recognize patterns, make predictions, and automate complex tasks in fields such as healthcare, finance, and autonomous systems.
⚠️ **Consider revising your research to provide new insights.**
